# Lección 9 · Heurísticas: buenas soluciones rápido, y cómo saber qué tan buenas son

**Qué vas a aprender**
- **Heurística constructiva**: armar una solución con una regla simple (EDD, WSPT, ATC).
- **Búsqueda local**: mejorar con cambios chicos hasta que ningún vecino sea mejor. Qué es un **óptimo local**.
- **Simulated annealing** (SA): aceptar a veces una solución peor para salir de un óptimo local. Temperatura y enfriamiento.
- **Warm start**: darle la mejor solución heurística al solver como pista.
- **Límite de tiempo** y **gap**: medir la heurística contra el óptimo (instancias chicas) y contra la cota (instancias grandes).
- Reportar una metaheurística con honestidad: varias semillas, media y peor caso.
- Escribir cada modelo en **notación matemática** y leerla símbolo por símbolo.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Hoy vuelve **CP-SAT** (lección 7) como solver exacto de referencia. Mismas versiones fijas que en la lección 7: OR-Tools y `highspy` conviven así.

In [ ]:
%pip install -q pyomo "highspy==1.14.0" "ortools==9.15.6755"

In [ ]:
import io
import itertools
import logging
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from ortools.sat.python import cp_model
from pyomo.repn import generate_standard_repn

SOLVER = pyo.SolverFactory("appsi_highs")
logging.getLogger("pyomo.contrib.appsi").setLevel(logging.ERROR)


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"


def tamano(m):
    # El tamaño del modelo: variables, restricciones y no ceros (coeficientes distintos de 0)
    variables = list(m.component_data_objects(pyo.Var))
    restricciones = list(m.component_data_objects(pyo.Constraint, active=True))
    return {"variables": len(variables),
            "binarias": sum(v.is_binary() for v in variables),
            "restricciones": len(restricciones),
            "no_ceros": sum(sum(c != 0 for c in generate_standard_repn(r.body).linear_coefs) for r in restricciones)}

## 1. Dónde quedamos
En la lección 7, el banco de prueba de presión tenía 4 trabajos y el MILP con big-M daba una cota LP de 0. CP-SAT resolvía el taller en milisegundos.
Pero no siempre tienes un solver que llegue: hay reglas raras del negocio, instancias enormes o 2 segundos para responder.
Entonces se usa una **heurística**: un método que da una buena solución rápido, **sin garantía** de que sea la mejor.

**Analogía.** Bajar del Kasprowy Wierch en la niebla, sin mapa. Solo ves el suelo alrededor.
- **Regla constructiva:** "sigue el sendero marcado". Llegas abajo, no necesariamente por el camino más corto.
- **Búsqueda local:** en cada paso, da el paso que más baja. Si caes en un hoyo (un valle chico), te quedas: todo alrededor sube. Eso es un **óptimo local**.
- **Simulated annealing:** a veces subes un poco a propósito para salir del hoyo. Al principio subes seguido; al final, casi nunca.

La pregunta de hoy no es solo "¿qué heurística?", sino **"¿cómo sé que es buena?"**.

## 2. El banco de prueba, ahora con prioridades
El banco de prueba de presión tiene 8 trabajos. Cada uno tiene horas, una entrega y un **peso**: la prioridad del cliente.
Una hora de retraso de un cliente con peso 4 cuesta 4 veces lo que una de peso 1. `J2` ya está atrasado: el cliente lo quería para la hora 0.

**Las 3 piezas:**
- **Decido:** el orden de los 8 trabajos en el banco.
- **Objetivo:** minimizar el **retraso ponderado** total: $\sum_j w_j\, T_j$.
- **Restricciones:** un trabajo a la vez, sin interrupciones.

En la notación de los libros de scheduling este problema se llama $1 \,\Vert\, \sum w_j T_j$: **una** máquina, sin condiciones extra, retraso ponderado.
Es NP-difícil: no se conoce un método exacto que escale bien.

In [ ]:
CHICO = pd.read_csv(io.StringIO('''trabajo,horas,peso,entrega
J0,8,1,18
J1,1,1,19
J2,2,2,0
J3,3,3,3
J4,2,4,12
J5,8,3,10
J6,8,2,23
J7,6,1,13
'''))


def costo(orden, p, w, d):
    # orden: posiciones de los trabajos en la secuencia. Devuelve el retraso ponderado total.
    fin = np.cumsum(p[orden])
    return int((w[orden] * np.maximum(0, fin - d[orden])).sum())


def datos(df):
    return df["horas"].to_numpy(), df["peso"].to_numpy(), df["entrega"].to_numpy()


p, w, d = datos(CHICO)
print("Orden de la tabla:", costo(np.arange(8), p, w, d), "horas ponderadas de retraso")
CHICO

### En notación
Una solución es una **permutación** $\pi = (\pi_1, \dots, \pi_n)$: $\pi_k$ es el trabajo que va en la posición $k$.
El fin del trabajo en la posición $k$ es la suma de las horas de los que van antes y la suya.

$$
\begin{aligned}
\min_{\pi \in \Pi_n} \quad & \sum_{k=1}^{n} w_{\pi_k}\, \max\Big\{0,\ C_{\pi_k} - d_{\pi_k}\Big\} \\
\text{con} \quad & C_{\pi_k} = \sum_{i=1}^{k} p_{\pi_i}
\end{aligned}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\pi$, $\pi_k$ | "pi", "pi de k" | una secuencia; el trabajo en la posición $k$ | `orden`, `orden[k]` |
| $\Pi_n$ | "pi mayúscula de n" | todas las permutaciones de $n$ trabajos: $n!$ | `itertools.permutations(range(n))` |
| $C_{\pi_k} = \sum_{i \le k} p_{\pi_i}$ | "suma acumulada" | hora en que termina el $k$-ésimo trabajo | `np.cumsum(p[orden])` |
| $w_j$ | "peso de j" | prioridad del cliente | `peso` |
| $1 \,\Vert\, \sum w_j T_j$ | "uno, barra barra, suma de w T" | notación de Graham: máquinas · condiciones · objetivo | — |

Las heurísticas trabajan con $\pi$ directo: no necesitan variables ni restricciones. El MILP (sección 2.3) necesita traducir $\pi$ a binarias.

### 2.1 Fuerza bruta
$8! = 40.320$ órdenes. Python los prueba en menos de un segundo.

In [ ]:
inicio = time.perf_counter()
todos = sorted((costo(np.array(o), p, w, d), o) for o in itertools.permutations(range(8)))
OPTIMO_CHICO = todos[0][0]
print(f"Órdenes: {len(todos):,} en {time.perf_counter() - inicio:.1f} s".replace(",", "."))
print("Óptimo:", OPTIMO_CHICO, "| órdenes óptimos:", sum(c == OPTIMO_CHICO for c, _ in todos), "| el peor:", todos[-1][0])
print("Un orden óptimo:", " → ".join(CHICO["trabajo"].iloc[list(todos[0][1])]))

**Lectura:**
- El óptimo es **64** horas ponderadas de retraso. Hay 2 órdenes que lo logran. El peor cuesta 350: el orden importa 5 veces.
- Con 12 trabajos serían $12! \approx 479$ millones de órdenes: unos 40 minutos. Con 15, más de un mes. La fuerza bruta solo sirve para **medir** heurísticas en instancias chicas.

### 2.2 Reglas de prioridad: construir en una pasada
Una **regla de prioridad** ordena los trabajos por un número. Se calcula en un instante.
- **EDD** (*earliest due date*): primero el que vence antes. Ignora pesos y duraciones.
- **WSPT** (*weighted shortest processing time*): primero el de mayor $w_j / p_j$, "más prioridad por hora de banco". Es óptima si no hubiera entregas ($\sum w_j C_j$).
- **ATC** (*apparent tardiness cost*): mezcla las dos. En cada momento $t$ elige el trabajo con mayor índice

$$
I_j(t) = \frac{w_j}{p_j}\, \exp\left( -\frac{\max\{0,\ d_j - p_j - t\}}{k\, \bar{p}} \right)
$$

  El primer factor es WSPT. El segundo baja la prioridad de los trabajos con **holgura** (los que aún tienen tiempo). $\bar{p}$ es la duración media y $k \approx 2$ es un parámetro.

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $w_j / p_j$ | "w sobre p" | prioridad por hora de banco | `w / p` |
| $d_j - p_j - t$ | "holgura" | cuánto puede esperar el trabajo antes de atrasarse | `d - p - t` |
| $\exp(\cdot)$ | "exponencial" | $e^x$: con mucha holgura, el índice se acerca a 0 | `np.exp()` |
| $\bar{p}$ | "p barra" | duración media | `p.mean()` |

In [ ]:
def edd(p, w, d):
    return np.argsort(d, kind="stable")


def wspt(p, w, d):
    return np.argsort(-w / p, kind="stable")


def atc(p, w, d, k=2.0):
    t, quedan, orden = 0, list(range(len(p))), []
    while quedan:
        q = np.array(quedan)
        indice = w[q] / p[q] * np.exp(-np.maximum(0, d[q] - p[q] - t) / (k * p.mean()))
        j = q[indice.argmax()]
        orden.append(j)
        quedan.remove(j)
        t += p[j]
    return np.array(orden)


REGLAS = {"EDD": edd, "WSPT": wspt, "ATC": atc}
for nombre, regla in REGLAS.items():
    o = regla(p, w, d)
    c = costo(o, p, w, d)
    print(f"{nombre:>4}: {c:>3} | {100 * (c - OPTIMO_CHICO) / OPTIMO_CHICO:>4.0f} % sobre el óptimo | {' → '.join(CHICO['trabajo'].iloc[o])}")

**Lectura:**
- EDD da 91 (42 % sobre el óptimo) y WSPT 82 (28 %). Cada una mira solo un lado del problema.
- **ATC da 64: el óptimo.** Mezclar prioridad y urgencia funciona muy bien en esta instancia. No siempre llega; sí casi siempre queda cerca.

### 2.3 El MILP, como referencia
Es el modelo disyuntivo de la lección 7, con pesos. Sirve para comprobar el óptimo en instancias chicas.

$$
\begin{aligned}
\min \quad & \sum_{j \in J} w_j\, T_j \\
\text{s.a.} \quad & T_j \ge S_j + p_j - d_j && \forall\, j \in J \\
& S_j \ge S_i + p_i - M\,(1 - y_{ij}) && \forall\, i < j \\
& S_i \ge S_j + p_j - M\, y_{ij} && \forall\, i < j \\
& S_j \ge 0, \quad T_j \ge 0, \quad y_{ij} \in \{0, 1\}
\end{aligned}
$$

In [ ]:
def una_maquina_milp(df, limite=None):
    J = list(range(len(df)))
    p, w, d = datos(df)
    M = int(p.sum())
    m = pyo.ConcreteModel()
    m.PARES = pyo.Set(initialize=list(itertools.combinations(J, 2)), dimen=2)
    m.S = pyo.Var(J, within=pyo.NonNegativeReals)
    m.T = pyo.Var(J, within=pyo.NonNegativeReals)
    m.y = pyo.Var(m.PARES, within=pyo.Binary)
    m.obj = pyo.Objective(expr=sum(int(w[j]) * m.T[j] for j in J), sense=pyo.minimize)
    m.retraso = pyo.Constraint(J, rule=lambda m, j: m.T[j] >= m.S[j] + int(p[j]) - int(d[j]))
    m.antes = pyo.Constraint(m.PARES, rule=lambda m, i, j: m.S[j] >= m.S[i] + int(p[i]) - M * (1 - m.y[i, j]))
    m.despues = pyo.Constraint(m.PARES, rule=lambda m, i, j: m.S[i] >= m.S[j] + int(p[j]) - M * m.y[i, j])
    inicio = time.perf_counter()
    SOLVER.solve(m, timelimit=limite)
    m.segundos = time.perf_counter() - inicio
    return m


mm = una_maquina_milp(CHICO)
print(f"MILP: {pyo.value(mm.obj):.0f} en {mm.segundos:.2f} s")
print("Tamaño:", tamano(mm))

**Lectura:**
- El MILP confirma el óptimo: **64**. Con 8 trabajos son $\binom{8}{2} = 28$ binarias.

**Los dos tamaños.**
- Modelo: $n(n-1)/2$ binarias, $O(n^2)$. Con 200 trabajos, 19.900.
- Espacio de búsqueda: $n!$. Con 200 trabajos, $200! \approx 10^{375}$.
- Una regla de prioridad no recorre el espacio: hace $n$ elecciones, $O(n^2)$ cuentas con ATC. Por eso es instantánea.

## 3. Búsqueda local: mejorar con cambios chicos
Partimos de una solución y probamos **vecinos**: soluciones que se obtienen con un cambio chico.
La **vecindad** de hoy es el **intercambio** (*swap*): cambiar de lugar dos trabajos. Hay $\binom{n}{2}$ vecinos.
Si un vecino es mejor, nos movemos y empezamos otra vez. Si ninguno es mejor, paramos: es un **óptimo local**.

$$
\mathcal{N}(\pi) = \{\, \pi' : \pi' \text{ sale de } \pi \text{ intercambiando dos posiciones} \,\}, \qquad \lvert \mathcal{N}(\pi) \rvert = \binom{n}{2}
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\mathcal{N}(\pi)$ | "vecindad de pi" | las soluciones a un cambio de distancia | `itertools.combinations(range(n), 2)` |
| $\pi'$ | "pi prima" | un vecino | `vecino` |
| óptimo local | | ningún vecino es mejor; puede haber algo mejor más lejos | — |

In [ ]:
def busqueda_local(orden, p, w, d):
    # Primera mejora: recorre los intercambios y se mueve al primero que mejora. Para cuando una pasada entera no mejora.
    orden, c, pasos, mejoro = orden.copy(), costo(orden, p, w, d), 0, True
    while mejoro:
        mejoro = False
        for i, k in itertools.combinations(range(len(orden)), 2):
            vecino = orden.copy()
            vecino[i], vecino[k] = vecino[k], vecino[i]
            c_vecino = costo(vecino, p, w, d)
            if c_vecino < c:
                orden, c, pasos, mejoro = vecino, c_vecino, pasos + 1, True
    return orden, c, pasos


for nombre, regla in REGLAS.items():
    o, c, pasos = busqueda_local(regla(p, w, d), p, w, d)
    print(f"Desde {nombre:>4}: {costo(regla(p, w, d), p, w, d):>3} → {c:>3} en {pasos} movimientos")

**Lectura:**
- Desde EDD y desde WSPT, la búsqueda local llega al **óptimo, 64**, en pocos movimientos. Desde ATC no se mueve: ya estaba en el óptimo.
- En instancias chicas los óptimos locales suelen ser buenos. En grandes, la búsqueda se queda en el primer "hoyo" que encuentra.

## 4. Simulated annealing: a veces, aceptar algo peor
**Simulated annealing** (recocido simulado) viene de la metalurgia: se calienta el metal y se enfría despacio para que los átomos encuentren una forma estable.

En cada iteración:
1. Elige un vecino **al azar** (un intercambio al azar).
2. Si es mejor, acéptalo. Si es peor por $\Delta$, acéptalo con probabilidad $e^{-\Delta / T}$.
3. Baja la temperatura: $T \leftarrow \alpha\, T$, con $\alpha$ un poco menor que 1.

Con $T$ alta se acepta casi todo: explora. Con $T$ baja casi nada peor: se vuelve búsqueda local. Guarda siempre **la mejor** que haya visto.

$$
P(\text{aceptar}) = \begin{cases} 1 & \text{si } \Delta \le 0 \\ e^{-\Delta / T} & \text{si } \Delta > 0 \end{cases}
\qquad T_{k+1} = \alpha\, T_k
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\Delta$ | "delta" | cuánto empeora el vecino: $\text{costo}(\pi') - \text{costo}(\pi)$ | `c_vecino - c` |
| $T$, $T_0$ | "temperatura", "temperatura inicial" | en unidades del costo: horas ponderadas | `T`, `T0` |
| $e^{-\Delta / T}$ | "e a la menos delta sobre T" | con $\Delta = T$, se acepta 37 % de las veces | `math.exp(-delta / T)` |
| $\alpha$ | "alfa" | enfriamiento por iteración | `alfa` |

In [ ]:
def recocido(orden, p, w, d, iters=30_000, T0=10.0, semilla=0, evaluar=None):
    evaluar = evaluar or costo  # la función que mide un plan (en el ejercicio 3 la cambias)
    r = np.random.default_rng(semilla)
    n = len(orden)
    actual, c = orden.copy(), evaluar(orden, p, w, d)
    mejor, c_mejor, T = actual.copy(), c, T0
    alfa = 0.001 ** (1 / iters)  # al final, T vale 0,1 % de T0
    for _ in range(iters):
        i, k = r.choice(n, 2, replace=False)
        vecino = actual.copy()
        vecino[i], vecino[k] = vecino[k], vecino[i]
        c_vecino = evaluar(vecino, p, w, d)
        delta = c_vecino - c
        if delta <= 0 or r.random() < math.exp(-delta / T):
            actual, c = vecino, c_vecino
            if c < c_mejor:
                mejor, c_mejor = actual.copy(), c
        T *= alfa
    return mejor, c_mejor


inicio = time.perf_counter()
resultados_sa = [recocido(edd(p, w, d), p, w, d, iters=5_000, semilla=s)[1] for s in range(5)]
print("SA desde EDD, 5 semillas:", resultados_sa, f"| {(time.perf_counter() - inicio) / 5:.2f} s cada una")

**Lectura:**
- Las 5 corridas llegan al **óptimo, 64**. En esta instancia todo funciona: es chica.
- El SA tiene **perillas**: $T_0$, $\alpha$, iteraciones, vecindad. Elegirlas es parte del trabajo (ejercicio 2).

## 5. El modelo crece: de 10 a 200 trabajos
Generamos instancias al azar (semilla fija): de 1 a 9 horas, pesos de 1 a 5 y entregas apretadas.
Para cada tamaño probamos una **cadena**, como se hace en la práctica:
1. Las 3 reglas.
2. **Búsqueda local** desde ATC.
3. **SA** desde la búsqueda local, con 5 semillas.
4. **CP-SAT** con 10 segundos, solo y con la mejor heurística como **pista** (*warm start*).

CP-SAT da además una **cota**: ninguna secuencia baja de ahí. El **gap** contra la cota dice, en el peor caso, cuánto le falta a la mejor solución.

In [ ]:
def instancia(n, semilla=1):
    rng = np.random.default_rng(semilla)
    horas = rng.integers(1, 10, n)
    return pd.DataFrame({"trabajo": [f"J{j:03d}" for j in range(n)], "horas": horas, "peso": rng.integers(1, 6, n),
                         "entrega": rng.integers(0, int(horas.sum() * 0.7) + 1, n)})


def una_maquina_cp(df, limite=10, pista=None):
    p, w, d = datos(df)
    n, H = len(p), int(p.sum())
    m = cp_model.CpModel()
    ini = [m.new_int_var(0, H, f"ini_{j}") for j in range(n)]
    m.add_no_overlap([m.new_fixed_size_interval_var(ini[j], int(p[j]), f"int_{j}") for j in range(n)])
    T = [m.new_int_var(0, H, f"T_{j}") for j in range(n)]
    for j in range(n):
        m.add(T[j] >= ini[j] + int(p[j]) - int(d[j]))
    m.minimize(sum(int(w[j]) * T[j] for j in range(n)))
    if pista is not None:  # warm start: los inicios de la secuencia heurística
        fin = np.cumsum(p[pista])
        for j, f in zip(pista, fin):
            m.add_hint(ini[j], int(f - p[j]))
            m.add_hint(T[j], max(0, int(f - d[j])))
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = limite
    solver.parameters.num_workers = 8  # 8 estrategias en paralelo; con menos, la cota sale mucho peor
    estado = solver.solve(m)
    return {"estado": solver.status_name(estado), "valor": solver.objective_value, "cota": solver.best_objective_bound}


print(instancia(200).describe().loc[["mean", "min", "max"]].round(1))

La celda de escala tarda alrededor de 1 minuto. CP-SAT corre con 8 *workers* (estrategias en paralelo): con menos, la cota sale mucho peor.
Sus resultados cambian un poco con el computador: Colab tiene 2 núcleos, así que cada *worker* recibe menos tiempo.

In [ ]:
filas, mejores = [], {}
for n in [10, 20, 50, 200]:
    df_n = instancia(n)
    p_n, w_n, d_n = datos(df_n)
    fila = {"trabajos": n, **{r: costo(f(p_n, w_n, d_n), p_n, w_n, d_n) for r, f in REGLAS.items()}}
    inicio = time.perf_counter()
    o_bl, fila["búsqueda_local"], _ = busqueda_local(atc(p_n, w_n, d_n), p_n, w_n, d_n)
    fila["bl_s"] = round(time.perf_counter() - inicio, 2)
    inicio = time.perf_counter()
    sa = [recocido(o_bl, p_n, w_n, d_n, semilla=s) for s in range(5)]
    fila["sa_s"] = round((time.perf_counter() - inicio) / 5, 2)
    fila["SA_mejor"], fila["SA_peor"] = min(c for _, c in sa), max(c for _, c in sa)
    mejor_heur = min(sa, key=lambda t: t[1])[0]
    solo = una_maquina_cp(df_n)
    con_pista = una_maquina_cp(df_n, pista=mejor_heur)
    fila.update({"CP_solo": round(solo["valor"]), "CP_pista": round(con_pista["valor"]), "estado": con_pista["estado"],
                 "cota": round(max(solo["cota"], con_pista["cota"]))})
    mejor = min(fila["SA_mejor"], fila["CP_pista"], fila["CP_solo"])
    fila["gap_%"] = round(100 * (mejor - fila["cota"]) / mejor, 1)
    filas.append(fila)
    mejores[n] = (df_n, mejor_heur)
escala = pd.DataFrame(filas).set_index("trabajos")
escala

**Cómo se lee:**
- **Las reglas:** ATC le gana a EDD y a WSPT en todos los tamaños, y por mucho: con 200 trabajos, 39.345 contra 77.765 (WSPT) y 109.384 (EDD).
- **Búsqueda local:** mejora ATC entre 1,5 % y 9 %, en menos de un segundo.
- **SA desde la búsqueda local:** con 20 trabajos llega al óptimo (526) que la búsqueda local no vio. Con 50 baja de 1.598 a 1.576. Con 200 no se mueve:
  30.000 intercambios al azar son pocos cuando hay ~20.000 vecinos y casi todos empeoran.
- **CP-SAT solo, 10 s:** con 10 y 20 trabajos demuestra el óptimo (`OPTIMAL`). Con 50 y 200, su solución es **peor que ATC**, una regla de una línea: con 200, unas 62.000 contra 39.345.
  (CP-SAT en paralelo no es determinista: la solución a los 10 s cambia un poco en cada corrida. La cota y el óptimo demostrado, casi no.)
- **CP-SAT con pista:** en 10 s no mejora la pista, pero la conserva. Y aporta lo que ninguna heurística da: la **cota**.
- **El gap:** con 200 trabajos, la mejor solución está a lo más a **5,9 %** del óptimo. Con 50, a unos 38 %: ahí la cota es floja y no sabemos si el plan es casi óptimo o no.
  El gap es lo **peor** que puede pasar, no lo que pasa.

**Los dos tamaños, en la práctica.** El espacio de búsqueda pasa de $10! \approx 3{,}6 \cdot 10^6$ a $200! \approx 10^{375}$.
Las reglas crecen $O(n^2)$; una pasada de búsqueda local, $O(n^3)$ ($n^2$ vecinos × $n$ para evaluar cada uno); el SA, lo que le des de tiempo.

### 5.1 Reportar con honestidad
Una metaheurística es aleatoria. Una sola corrida puede tener suerte. Se reporta la **media**, el **peor caso** y la distancia a una referencia.

In [ ]:
df50 = instancia(50)
p50, w50, d50 = datos(df50)
inicio50 = busqueda_local(atc(p50, w50, d50), p50, w50, d50)[0]
corridas = pd.Series([recocido(inicio50, p50, w50, d50, semilla=s)[1] for s in range(10)], name="SA")
ref = min(escala.loc[50, "CP_pista"], corridas.min())  # la mejor solución conocida
print(f"10 semillas: media {corridas.mean():.0f}, mejor {corridas.min()}, peor {corridas.max()}")
print(f"Contra la mejor conocida ({ref}): media {100 * (corridas.mean() - ref) / ref:+.1f} %, peor {100 * (corridas.max() - ref) / ref:+.1f} %")

**Lectura:**
- Con las semillas 5 a 9 apareció una solución mejor: **1.562**. Ahora es la mejor conocida. Las 5 corridas de la tabla de escala no la habían visto.
- Con 10 semillas el SA queda en promedio a 0,9 % de esa solución, y en el peor caso a 1,4 %.
- Así se presenta: *"en 10 corridas, a 0,9 % en promedio y 1,4 % en el peor caso de la mejor solución conocida"*. No "el SA encontró 1.562": esa fue la corrida con suerte.

### 5.2 La solución de 200 trabajos, de vuelta a tablas

In [ ]:
df200, orden200 = mejores[200]
p2, w2, d2 = datos(df200)
plan200 = df200.iloc[orden200].reset_index(drop=True)
plan200["fin"] = plan200["horas"].cumsum()
plan200["retraso"] = (plan200["fin"] - plan200["entrega"]).clip(lower=0)
plan200["retraso_ponderado"] = plan200["peso"] * plan200["retraso"]
print("Retraso ponderado total:", plan200["retraso_ponderado"].sum(), "| trabajos a tiempo:", (plan200["retraso"] == 0).sum(), "de 200")
plan200.groupby("peso").agg(trabajos=("trabajo", "count"), a_tiempo=("retraso", lambda r: (r == 0).sum()),
                            retraso_medio_h=("retraso", "mean"), posicion_media=("fin", "mean")).round(1)

In [ ]:
# Top 5: los trabajos que más retraso ponderado aportan
plan200.nlargest(5, "retraso_ponderado")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
colores = plt.cm.viridis(np.linspace(0, 0.9, 5))
for peso, grupo in plan200.groupby("peso"):
    ax.scatter(grupo["fin"], grupo["retraso"], s=12 + 6 * peso, color=colores[peso - 1], label=f"peso {peso}")
ax.set(xlabel="hora en que termina", ylabel="retraso (h)", title="200 trabajos: los de peso bajo se dejan para el final")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lectura para el jefe del taller:**
- La secuencia propuesta deja un retraso ponderado de **38.762 horas**. Con la regla actual del taller (EDD) serían 109.384: **casi 3 veces menos**.
- 89 de los 200 trabajos salen a tiempo. Los de peso 4 y 5 casi no se atrasan: 0,7 a 3,8 h en promedio.
- Los retrasos grandes son de peso 1 y 2, al final de la cola: el top 5 son trabajos de peso 2 y 8 a 9 horas, con unas 800 h de retraso. Es lo que el plan sacrifica, y hay que avisarles a esos clientes.
- No podemos demostrar que sea el óptimo, pero sí que **ninguna secuencia baja más de 5,9 %**. Para decidir, alcanza.
- Para el día a día sin computador, ATC solo queda a 1,5 % de este plan.

## 6. ¿Heurística o solver exacto?
Una sola regla para empezar:

> **Primero un solver exacto con límite de tiempo y la mejor heurística como pista. Heurística sola cuando el solver no cabe: reglas que no se linealizan, segundos para responder o tamaños enormes.**

| Situación | Herramienta | Por qué |
|---|---|---|
| Instancia chica o mediana, estructura clara | MILP o CP-SAT | da el óptimo o el gap |
| Instancia grande, el solver no encuentra buenas soluciones | heurística + solver con pista | la heurística da el punto de partida; el solver mejora y da cota |
| Responder en milisegundos (replanificar en línea) | regla de prioridad (ATC) | $O(n^2)$, sin solver |
| Reglas de negocio raras, costo no lineal, simulación dentro | SA u otra metaheurística | solo necesita **evaluar** un plan, no escribirlo como modelo |
| Hay que demostrar calidad | solver con cota | una heurística sola no sabe cuán lejos está |

**Siempre:** mide la heurística contra el óptimo en instancias chicas y contra la cota en las grandes.

## 7. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Heurística | Método rápido sin garantía de óptimo | reglas, búsqueda local, SA | heuristic |
| Regla de prioridad | Ordenar por un número | EDD, WSPT, ATC | dispatching rule |
| Heurística constructiva | Armar la solución paso a paso | ATC: 64, el óptimo | constructive heuristic |
| Vecindad | Soluciones a un cambio de distancia | intercambiar dos trabajos | neighborhood, swap move |
| Búsqueda local | Moverse al vecino mejor hasta que no haya | de 91 a 64 desde EDD | local search, hill climbing |
| Óptimo local | Ningún vecino es mejor | donde para la búsqueda local | local optimum |
| Metaheurística | Estrategia para salir de óptimos locales | SA | metaheuristic |
| Recocido simulado | Aceptar peores con probabilidad $e^{-\Delta/T}$ | $T_0 = 10$ | simulated annealing |
| Enfriamiento | Bajar la temperatura en cada iteración | $T \leftarrow \alpha T$ | cooling schedule |
| Arranque en caliente | Darle al solver una solución inicial | pista para CP-SAT | warm start, solution hint |
| Cota y gap | Lo mejor posible y la distancia a ello | 5,9 % con 200 trabajos | bound, optimality gap |
| Reporte honesto | Varias semillas: media y peor caso | 10 corridas | robustness across seeds |
| Notación de Graham | Máquinas · condiciones · objetivo | $1 \,\Vert\, \sum w_j T_j$ | three-field notation |

### Notación de hoy
| Símbolo | Significa | En la lección |
|---|---|---|
| $\pi$, $\pi_k$, $\Pi_n$ | secuencia, trabajo en la posición $k$, todas las secuencias | `orden` |
| $C_{\pi_k} = \sum_{i \le k} p_{\pi_i}$ | fin del trabajo en la posición $k$ | `np.cumsum` |
| $\sum_j w_j T_j$ | retraso ponderado | 64 en el banco |
| $1 \,\Vert\, \sum w_j T_j$ | notación de Graham | una máquina |
| $I_j(t)$ | índice de prioridad de ATC en el momento $t$ | `atc()` |
| $\exp(\cdot)$, $e^x$ | exponencial | ATC y SA |
| $\mathcal{N}(\pi)$, $\pi'$ | vecindad y vecino | intercambios |
| $\Delta$, $T$, $\alpha$ | empeoramiento, temperatura, enfriamiento | SA |
| $P(\text{aceptar}) = e^{-\Delta/T}$ | regla de aceptación de Metropolis | SA |
| $O(n^2)$, $O(n^3)$ | cómo crece el trabajo de un método | ATC, una pasada de búsqueda local |

## 8. Ejercicios
1. **Otra vecindad: insertar.** En vez de intercambiar dos trabajos, saca uno y ponlo en otra posición: `np.insert(np.delete(orden, i), k, orden[i])`.
   Escribe `busqueda_local_insertar` y compárala con la de intercambio en la instancia de 50 trabajos, desde ATC. ¿Cuál llega más abajo?
2. **La temperatura.** Corre `recocido` en la instancia de 50 desde ATC con `T0` = 0,01, 10 y 10.000 (3 semillas cada uno). Antes, adivina: ¿cuál es casi una búsqueda local? ¿Cuál camina al azar?
3. **Una regla rara del negocio.** El cliente de `J000` (instancia de 50) exige que su trabajo esté entre los 3 primeros. Agrégalo al SA sumando una multa de 10.000 si no se cumple.
   ¿Cuánto sube el retraso ponderado? ¿Cómo lo harías en el MILP?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"When would you use a heuristic instead of a MILP solver?"*
> *"When the exact model doesn't fit: rules that don't linearize, millisecond response times, or instances where the solver can't even find a good solution.
> But I rarely use a heuristic alone. In my single-machine weighted tardiness example, CP-SAT alone with 10 seconds was worse than the ATC dispatching rule on 200 jobs.
> With the heuristic solution as a hint, it kept that solution and gave a bound, so I could say the plan was within 6% of optimal.
> I always benchmark heuristics against the optimum on small instances and against the bound on large ones."*

Seguimiento: *"How does simulated annealing escape local optima, and how do you tune it?"*
> *"It accepts a worse neighbor with probability exp(-delta / T). Temperature starts high, so it explores, and decreases geometrically, so it ends as local search.
> The initial temperature should be in the units of the cost, around the size of a typical worsening move. Too low and it's plain hill climbing;
> too high and it's a random walk. I tune it on small instances with known optimum and report mean and worst case over several seeds."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
def busqueda_local_insertar(orden, p, w, d):
    orden, c, mejoro = orden.copy(), costo(orden, p, w, d), True
    while mejoro:
        mejoro = False
        for i, k in itertools.permutations(range(len(orden)), 2):
            vecino = np.insert(np.delete(orden, i), k, orden[i])
            c_vecino = costo(vecino, p, w, d)
            if c_vecino < c:
                orden, c, mejoro = vecino, c_vecino, True
    return orden, c


inicio_atc = atc(p50, w50, d50)
print("ATC:", costo(inicio_atc, p50, w50, d50))
print("Búsqueda local con intercambio:", busqueda_local(inicio_atc, p50, w50, d50)[1])
print("Búsqueda local con inserción:  ", busqueda_local_insertar(inicio_atc, p50, w50, d50)[1])

La inserción llega un poco más abajo: **1.593** contra 1.598. Su vecindad es más grande ($n(n-1)$ contra $n(n-1)/2$) y tiene movimientos que el intercambio no:
adelantar un trabajo urgente varias posiciones sin mandar otro al fondo. En scheduling se suelen usar las dos juntas.

In [ ]:
# Ejercicio 2
for T0 in [0.01, 10, 10_000]:
    res = [recocido(inicio_atc, p50, w50, d50, T0=T0, semilla=s)[1] for s in range(3)]
    print(f"T0 = {T0:>8}: {res}")

- $T_0 = 0{,}01$: casi nunca acepta algo peor. Es una búsqueda local con vecinos al azar: 1.576 a 1.593, según por dónde caiga.
- $T_0 = 10$: explora un poco al principio. 1.576 a 1.581.
- $T_0 = 10.000$: acepta casi todo durante casi toda la corrida. Camina al azar y **nunca** encuentra algo mejor que el inicio: 1.675, el ATC.
  Con $T$ tan alta, para cuando se enfría ya se acabaron las iteraciones.

**Regla:** $T_0$ del tamaño de un empeoramiento típico de un vecino (aquí, decenas de horas ponderadas). Se calibra en instancias con óptimo conocido.

In [ ]:
# Ejercicio 3
J000 = int(np.where(df50["trabajo"] == "J000")[0][0])


def costo_con_regla(orden, p, w, d):
    multa = 0 if J000 in orden[:3] else 10_000
    return costo(orden, p, w, d) + multa


print("J000 en el inicio (ATC + búsqueda local): posición", list(inicio50).index(J000) + 1)
orden_regla, _ = recocido(inicio50, p50, w50, d50, semilla=0, evaluar=costo_con_regla)
print("Con la regla: retraso ponderado", costo(orden_regla, p50, w50, d50), "| J000 en la posición", list(orden_regla).index(J000) + 1)
print("Sin la regla:", recocido(inicio50, p50, w50, d50, semilla=0)[1])

`J000` estaba en la posición 47. El SA con la multa lo sube a la **posición 3** y el retraso ponderado pasa de 1.576 a 1.593: **17 horas ponderadas**, el 1 %. Ese es el precio de la regla del cliente.
En el SA fue una línea: cambiar la función que mide el plan. En el MILP disyuntivo, "a lo más 2 trabajos antes de J000" es una restricción con las binarias de orden:
$\sum_{i \ne \text{J000}} [\,i \text{ antes de J000}\,] \le 2$, donde cada corchete es $y_{i,\text{J000}}$ o $1 - y_{\text{J000},i}$ según cómo esté guardado el par.
Con reglas más raras ("si J000 va antes de las 10 h, J007 no puede ir justo después") el MILP se llena de big-M; el SA sigue siendo una línea.